# Reproduce Experiment 5 — OTOC and Lyapunov exponent

Retrieves job `d7jaamkolufs73etbrfg` (OTOC F(t) at 13 times) and refits C(t) = 1 − F(t) = A·exp(λ_L t) for t ≤ 1. The MSS bound 2π/β_eff = 3.6034 is computed from the same SYK Hamiltonian in the experiment notebook.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
import numpy as np

service = QiskitRuntimeService(channel="ibm_cloud")

def check(name, value, paper, tol):
    ok = abs(value - paper) <= tol
    print(f"{name:32s} recomputed {value:9.4f}   paper {paper:9.4f}   {'OK' if ok else 'DIFFERENT'}")

In [ ]:
from scipy.optimize import curve_fit

T = np.array([0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0])
job = service.job("d7jaamkolufs73etbrfg")
r = job.result()
F = np.array([float(r[i].data.evs) for i in range(len(T))])
C = 1 - F
for t, f in zip(T, F):
    print(f"t = {t:4.1f}   F = {f:7.4f}")

m = T <= 1.0
(A, lam), cov = curve_fit(lambda t, A, l: A*np.exp(l*t), T[m], C[m], p0=[0.01, 2.0], maxfev=10000)
lam_err = np.sqrt(np.diag(cov))[1]
BOUND_MSS = 3.6034

In [ ]:
check("lambda_L", lam, 1.975, 0.0005)
check("lambda_L uncertainty", lam_err, 0.294, 0.0005)
check("% of MSS bound", 100*lam/BOUND_MSS, 54.8, 0.05)